# Exercise 2.4: STGNN with GRU on the Warsaw Bike-Sharing Graph

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/02_deep_learning/notebooks/exercise_2_4_warsaw_stgnn_gru_temporal_graph.ipynb)

This notebook uses the same Warsaw graph snapshots as Exercise 2.3, but changes the task to short spatio-temporal forecasting.

- Data source: Warsaw Bike-Sharing Daily Periods Graph Dataset for GNN, Season 2023, Mendeley Data, DOI: 10.17632/kzvdgfzk4w.1.
- Input: current station/weather/time features plus the previous five graph snapshots' station-level targets.
- Target: current station-level `log1p(incoming + outgoing trips)`.
- Models: Random Forest, XGBoost, GRU without graph edges, STGNN with GraphSAGE-GRU-GraphSAGE, and STGNN with shuffled edges.
- Colab runtime: choose `Runtime` -> `Change runtime type` -> `T4 GPU`; otherwise neural network training can be slow.

The train/validation/test split is chronological and uses an embargo gap, so raw graph snapshots used in one split do not overlap with another split.

## 1. Setup

PyTorch Geometric is installed only when running in Colab.

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install networkx scikit-learn xgboost matplotlib pandas torch-geometric

In [ ]:
from pathlib import Path
from datetime import datetime
import fnmatch
import math
import pickle
import random
import urllib.request
import warnings
import zipfile

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

## 2. Load Warsaw Graph Snapshots

The archive is the same course copy used in Exercise 2.3. The default uses the first contiguous half of the available snapshots to keep the notebook practical in Colab while preserving real `t-5` through `t-1` temporal order. Set `DATASET_FRACTION = 1.0` for the full dataset.

In [ ]:
GRAPH_PATTERN = "*.pt"
DATASET_FRACTION = 1 / 2
MAX_GRAPHS = None
HISTORY_STEPS = 5
EMBARGO_STEPS = HISTORY_STEPS

RF_TREES = 60
XGB_MAX_TREES = 300
XGB_EARLY_STOPPING_ROUNDS = 25
HGB_ITERATIONS = 100
GNN_EPOCHS = 200
GNN_PATIENCE = 20
BATCH_SIZE = 64

DAILY_PERIOD_ORDER = ["morning_peak", "midday", "afternoon_peak", "evening", "night"]
DATA_DIR = Path("/content/warsaw_gnn") if IN_COLAB else Path("data/warsaw_gnn")
ARCHIVE_FILE = DATA_DIR / "warsaw_gnn_dataset.zip"
SEAFILE_ARCHIVE_URL = "https://seafile.rlp.net/seafhttp/f/d0d006cf506b4ed79da8/?op=view"


def ensure_archive() -> Path:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE_FILE.exists():
        print(f"Downloading Warsaw GNN archive from Seafile to {ARCHIVE_FILE}...")
        urllib.request.urlretrieve(SEAFILE_ARCHIVE_URL, ARCHIVE_FILE)
    else:
        print(f"Using cached archive: {ARCHIVE_FILE}")
    return ARCHIVE_FILE


def graph_date_from_name(name: str) -> pd.Timestamp:
    day, month, year = Path(name).stem.split("_")[-3:]
    return pd.Timestamp(datetime(int(year), int(month), int(day)))


def graph_period_from_name(name: str) -> str:
    return "_".join(Path(name).stem.split("_")[:-3])


def graph_sort_key(name: str) -> tuple[pd.Timestamp, int, str]:
    period_order = {name: idx for idx, name in enumerate(DAILY_PERIOD_ORDER)}
    period = graph_period_from_name(name)
    return graph_date_from_name(name), period_order.get(period, len(period_order)), Path(name).name


def contiguous_sample(names: list[str], max_graphs: int | None) -> list[str]:
    if max_graphs is None or max_graphs >= len(names):
        return names
    return names[:max_graphs]


def selected_graph_count(total_graphs: int) -> int:
    if MAX_GRAPHS is not None:
        return min(MAX_GRAPHS, total_graphs)
    if DATASET_FRACTION >= 1.0:
        return total_graphs
    return max(HISTORY_STEPS + 10, int(round(total_graphs * DATASET_FRACTION)))


def graph_names_from_archive(archive_file: Path) -> tuple[list[str], pd.DataFrame, pd.DataFrame]:
    with zipfile.ZipFile(archive_file) as zf:
        all_pt_names = sorted([name for name in zf.namelist() if name.endswith(".pt")], key=graph_sort_key)
        matches = [name for name in all_pt_names if fnmatch.fnmatch(Path(name).name, GRAPH_PATTERN)]
        if not matches:
            raise FileNotFoundError(f"No graph files matching {GRAPH_PATTERN!r} were found in {archive_file}")
    selected = contiguous_sample(matches, selected_graph_count(len(matches)))
    all_dates = pd.Series([graph_date_from_name(name) for name in all_pt_names])
    selected_dates = pd.Series([graph_date_from_name(name) for name in selected])
    all_span_days = (all_dates.max() - all_dates.min()).days + 1
    selected_span_days = (selected_dates.max() - selected_dates.min()).days + 1
    summary = pd.DataFrame([
        {"scope": "all graph files in archive", "graph_snapshots": len(all_pt_names), "periods": ", ".join(sorted({graph_period_from_name(name) for name in all_pt_names})), "date_start": all_dates.min().date(), "date_end": all_dates.max().date(), "time_span_days": all_span_days, "share_of_all_snapshots": 1.0, "share_of_all_time_span": 1.0},
        {"scope": f"used in this notebook ({len(selected)}/{len(matches)} matching files)", "graph_snapshots": len(selected), "periods": ", ".join(sorted({graph_period_from_name(name) for name in selected})), "date_start": selected_dates.min().date(), "date_end": selected_dates.max().date(), "time_span_days": selected_span_days, "share_of_all_snapshots": len(selected) / len(all_pt_names), "share_of_all_time_span": selected_span_days / all_span_days},
    ])
    selected_table = pd.DataFrame({"graph_file": [Path(name).name for name in selected], "date": [graph_date_from_name(name).date() for name in selected], "period": [graph_period_from_name(name) for name in selected], "weekday": [graph_date_from_name(name).day_name() for name in selected]})
    return selected, summary, selected_table


archive_file = ensure_archive()
selected_names, coverage_summary, selected_graphs_table = graph_names_from_archive(archive_file)
observed_daily_periods = sorted({graph_period_from_name(name) for name in selected_names})
missing_daily_periods = [period for period in observed_daily_periods if period not in DAILY_PERIOD_ORDER]
if missing_daily_periods:
    DAILY_PERIOD_ORDER = [period for period in DAILY_PERIOD_ORDER if period in observed_daily_periods] + missing_daily_periods

display(coverage_summary)
display(selected_graphs_table.head(10))
display(selected_graphs_table.tail(10))
print(f"selected_graph_files={len(selected_names):,}, history_steps={HISTORY_STEPS}")
print(f"daily_period_order={DAILY_PERIOD_ORDER}")

## 3. Convert Snapshots Into Node Rows and Edge Tensors

The target is station-level total bike flow, transformed with `log1p`. Absolute coordinates are excluded from model features. Relative distance descriptors such as `d_city_cen` remain valid model features.

In [ ]:
def edge_weight(edge_data: dict) -> float:
    for key in ["trips_count", "trip_count", "count", "weight"]:
        if key in edge_data:
            try:
                return float(edge_data[key])
            except Exception:
                return 0.0
    return 1.0


def temporal_features_from_graph_file(graph_file: str) -> dict:
    graph_date = graph_date_from_name(graph_file)
    period = graph_period_from_name(graph_file)
    period_to_index = {name: idx for idx, name in enumerate(DAILY_PERIOD_ORDER)}
    if period not in period_to_index:
        raise ValueError(f"Unknown daily period {period!r}. Check DAILY_PERIOD_ORDER.")
    period_index = period_to_index[period]
    weekday_index = graph_date.weekday()
    day_of_year = graph_date.dayofyear
    return {
        "graph_date": graph_date.date(),
        "daily_period": period,
        "time_sin": math.sin(2 * math.pi * period_index / len(DAILY_PERIOD_ORDER)),
        "time_cos": math.cos(2 * math.pi * period_index / len(DAILY_PERIOD_ORDER)),
        "weekday_sin": math.sin(2 * math.pi * weekday_index / 7),
        "weekday_cos": math.cos(2 * math.pi * weekday_index / 7),
        "year_sin": math.sin(2 * math.pi * day_of_year / 365),
        "year_cos": math.cos(2 * math.pi * day_of_year / 365),
    }


def graph_to_node_frame(graph: nx.DiGraph, graph_file: str, graph_position: int) -> pd.DataFrame:
    graph_temporal_features = temporal_features_from_graph_file(graph_file)
    total_flow = {node: 0.0 for node in graph.nodes}
    in_flow = {node: 0.0 for node in graph.nodes}
    out_flow = {node: 0.0 for node in graph.nodes}
    for u, v, data in graph.edges(data=True):
        w = edge_weight(data)
        out_flow[u] += w
        in_flow[v] += w
        total_flow[u] += w
        total_flow[v] += w
    rows = []
    for node, attrs in graph.nodes(data=True):
        row = {"graph_position": graph_position, "graph_file": graph_file, "node_id": node, **graph_temporal_features, **dict(attrs)}
        row["target_total_trips"] = total_flow[node]
        row["target_in_trips"] = in_flow[node]
        row["target_out_trips"] = out_flow[node]
        rows.append(row)
    return pd.DataFrame(rows)


def process_graphs_streaming(archive_file: Path, selected_names: list[str]) -> tuple[pd.DataFrame, list[torch.Tensor], pd.DataFrame]:
    node_frames = []
    graph_edge_indices = []
    graph_slices = []
    node_offset = 0
    with zipfile.ZipFile(archive_file) as zf:
        for graph_position, archive_name in enumerate(selected_names):
            graph_file = Path(archive_name).name
            with zf.open(archive_name) as f:
                graph = pickle.load(f)
            if not isinstance(graph, nx.Graph):
                raise TypeError(f"Expected a NetworkX graph in {graph_file}, got {type(graph)!r}")
            graph = nx.DiGraph(graph)
            node_frame = graph_to_node_frame(graph, graph_file, graph_position)
            node_frames.append(node_frame)
            local_node_to_pos = {node: idx for idx, node in enumerate(graph.nodes())}
            src, dst = [], []
            for u, v in graph.edges():
                if u in local_node_to_pos and v in local_node_to_pos and u != v:
                    src_pos = local_node_to_pos[u]
                    dst_pos = local_node_to_pos[v]
                    src.extend([src_pos, dst_pos])
                    dst.extend([dst_pos, src_pos])
            if not src:
                raise ValueError(f"No usable edges were found in {graph_file}.")
            edge_index = torch.from_numpy(np.vstack([np.asarray(src, dtype=np.int64), np.asarray(dst, dtype=np.int64)])).long().contiguous()
            graph_edge_indices.append(edge_index)
            graph_slices.append({"graph_position": graph_position, "graph_file": graph_file, "row_start": node_offset, "row_end": node_offset + len(node_frame), "nodes": len(node_frame), "edge_index_columns": int(edge_index.shape[1])})
            node_offset += len(node_frame)
            if (graph_position + 1) % 100 == 0 or graph_position + 1 == len(selected_names):
                print(f"processed_graphs={graph_position + 1:,}/{len(selected_names):,}, node_rows={node_offset:,}")
            del graph
    nodes = pd.concat(node_frames, ignore_index=True)
    nodes["y_log_total_trips"] = np.log1p(nodes["target_total_trips"].astype(float))
    graph_slices = pd.DataFrame(graph_slices).set_index("graph_position", drop=False)
    return nodes, graph_edge_indices, graph_slices


nodes, graph_edge_indices, graph_slices = process_graphs_streaming(archive_file, selected_names)
print(f"node_rows={len(nodes):,}, graph_snapshots={nodes['graph_position'].nunique():,}, edge_index_columns={graph_slices['edge_index_columns'].sum():,}")

In [ ]:
def is_absolute_coordinate_feature(column: str) -> bool:
    lower = column.lower()
    exact_coordinate_names = {"lat", "lng", "lon", "latitude", "longitude", "x", "y"}
    return lower in exact_coordinate_names or "centroid_latitude" in lower or "centroid_longitude" in lower


exclude_exact = {"graph_position", "graph_file", "graph_date", "daily_period", "node_id", "target_total_trips", "target_in_trips", "target_out_trips", "y_log_total_trips"}
exclude_contains = ["trip", "flow", "target"]
candidate_features = []
excluded_coordinate_features = []
for column in nodes.columns:
    if column in exclude_exact or any(term in column.lower() for term in exclude_contains):
        continue
    if is_absolute_coordinate_feature(column):
        excluded_coordinate_features.append(column)
        continue
    numeric = pd.to_numeric(nodes[column], errors="coerce")
    if numeric.notna().sum() >= max(5, int(0.2 * len(nodes))) and numeric.nunique(dropna=True) > 1:
        nodes[column] = numeric
        candidate_features.append(column)

if not candidate_features:
    raise ValueError("No numeric node features were found.")

feature_frame = nodes[candidate_features].copy().replace([np.inf, -np.inf], np.nan)
feature_frame = feature_frame.fillna(feature_frame.median(numeric_only=True)).fillna(0)
print(f"Using {len(candidate_features)} current-snapshot node features:")
print(candidate_features)
print("Excluded absolute coordinate features:")
print(excluded_coordinate_features)
nodes[["graph_position", "graph_file", "node_id", "target_total_trips", "y_log_total_trips"] + candidate_features[:8]].head()

## 4. Build Non-Overlapping Temporal Samples

A sample predicts graph snapshot `t` from snapshots `t-5`, `t-4`, `t-3`, `t-2`, and `t-1`. The split is chronological. An embargo of five snapshots is inserted between train, validation, and test so no raw graph snapshot is reused across splits.

In [ ]:
def split_temporal_targets(n_graphs: int, history_steps: int = HISTORY_STEPS, embargo_steps: int = EMBARGO_STEPS) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    possible_targets = np.arange(history_steps, n_graphs)
    n_targets = len(possible_targets)
    train_count = max(1, int(0.6 * n_targets))
    val_count = max(1, int(0.2 * n_targets))
    train_targets = possible_targets[:train_count]
    val_start = train_targets[-1] + embargo_steps + 1
    val_targets = possible_targets[possible_targets >= val_start][:val_count]
    if len(val_targets) == 0:
        raise ValueError("Not enough snapshots for validation after the embargo gap.")
    test_start = val_targets[-1] + embargo_steps + 1
    test_targets = possible_targets[possible_targets >= test_start]
    if len(test_targets) == 0:
        raise ValueError("Not enough snapshots for testing after the embargo gap.")
    return train_targets, val_targets, test_targets


def raw_positions_used(target_positions: np.ndarray, history_steps: int = HISTORY_STEPS) -> set[int]:
    used = set()
    for target_position in target_positions:
        used.update(range(int(target_position) - history_steps, int(target_position) + 1))
    return used


def row_indices_for_targets(target_positions: np.ndarray) -> np.ndarray:
    chunks = []
    for target_position in target_positions:
        row = graph_slices.loc[int(target_position)]
        chunks.append(np.arange(int(row["row_start"]), int(row["row_end"])))
    return np.concatenate(chunks)


train_targets, val_targets, test_targets = split_temporal_targets(len(graph_slices))
train_raw, val_raw, test_raw = raw_positions_used(train_targets), raw_positions_used(val_targets), raw_positions_used(test_targets)
assert train_raw.isdisjoint(val_raw) and train_raw.isdisjoint(test_raw) and val_raw.isdisjoint(test_raw)
train_idx, val_idx, test_idx = row_indices_for_targets(train_targets), row_indices_for_targets(val_targets), row_indices_for_targets(test_targets)

split_summary = pd.DataFrame([
    {"split": "train", "target_snapshots": len(train_targets), "raw_snapshots_used": len(train_raw), "target_start": int(train_targets[0]), "target_end": int(train_targets[-1]), "node_rows": len(train_idx)},
    {"split": "validation", "target_snapshots": len(val_targets), "raw_snapshots_used": len(val_raw), "target_start": int(val_targets[0]), "target_end": int(val_targets[-1]), "node_rows": len(val_idx)},
    {"split": "test", "target_snapshots": len(test_targets), "raw_snapshots_used": len(test_raw), "target_start": int(test_targets[0]), "target_end": int(test_targets[-1]), "node_rows": len(test_idx)},
])
display(split_summary)
print(f"raw train/validation overlap={len(train_raw & val_raw)}, train/test overlap={len(train_raw & test_raw)}, validation/test overlap={len(val_raw & test_raw)}")

In [ ]:
X = feature_frame.to_numpy(dtype=np.float32)
y = nodes["y_log_total_trips"].to_numpy(dtype=np.float32)
scaler = StandardScaler()
X_scaled = scaler.fit(X[train_idx]).transform(X).astype(np.float32)
y_mean = float(y[train_idx].mean())
y_std = float(y[train_idx].std() + 1e-8)
y_scaled = ((y - y_mean) / y_std).astype(np.float32)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    return {"rmse": float(math.sqrt(mse)), "mae": float(mean_absolute_error(y_true, y_pred)), "r2": float(r2_score(y_true, y_pred))}

print(f"train_rows={len(train_idx):,}, validation_rows={len(val_idx):,}, test_rows={len(test_idx):,}")

## 5. Create Lagged Graph Samples

Each PyG `Data` object contains one target snapshot: current features `x`, previous five target values `history_y`, current graph edges `edge_index`, and standardized target `y`.

In [ ]:
def make_window_data(target_position: int, shuffled_edges: bool = False) -> Data:
    target_slice = graph_slices.loc[int(target_position)]
    target_rows = np.arange(int(target_slice["row_start"]), int(target_slice["row_end"]))
    target_nodes = nodes.iloc[target_rows]["node_id"].to_numpy()
    history_columns = []
    for lag in range(HISTORY_STEPS, 0, -1):
        history_position = int(target_position) - lag
        history_slice = graph_slices.loc[history_position]
        history_rows = np.arange(int(history_slice["row_start"]), int(history_slice["row_end"]))
        history_series = nodes.iloc[history_rows].set_index("node_id")["y_log_total_trips"]
        history_values = pd.Series(target_nodes).map(history_series).to_numpy(dtype=np.float32)
        if np.isnan(history_values).any():
            raise ValueError(f"Missing history for graph_position={target_position}, lag={lag}")
        history_columns.append(history_values)
    history_raw = np.column_stack(history_columns).astype(np.float32)
    history_scaled = ((history_raw - y_mean) / y_std).astype(np.float32)
    edge_index = graph_edge_indices[int(target_position)]
    if shuffled_edges:
        rng = np.random.default_rng(SEED + int(target_position))
        node_permutation = torch.from_numpy(rng.permutation(len(target_rows))).long()
        edge_index = node_permutation[edge_index]
    data = Data(x=torch.tensor(X_scaled[target_rows], dtype=torch.float32), history_y=torch.tensor(history_scaled, dtype=torch.float32), edge_index=edge_index, y=torch.tensor(y_scaled[target_rows].reshape(-1, 1), dtype=torch.float32))
    data.y_raw = torch.tensor(y[target_rows].reshape(-1, 1), dtype=torch.float32)
    data.graph_position = int(target_position)
    data.graph_file = target_slice["graph_file"]
    return data


def make_dataset(target_positions: np.ndarray, shuffled_edges: bool = False) -> list[Data]:
    return [make_window_data(int(target_position), shuffled_edges=shuffled_edges) for target_position in target_positions]


train_data, val_data, test_data = make_dataset(train_targets), make_dataset(val_targets), make_dataset(test_targets)
train_data_shuffled, val_data_shuffled, test_data_shuffled = make_dataset(train_targets, True), make_dataset(val_targets, True), make_dataset(test_targets, True)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
train_loader_shuffled = DataLoader(train_data_shuffled, batch_size=BATCH_SIZE, shuffle=True)
val_loader_shuffled = DataLoader(val_data_shuffled, batch_size=BATCH_SIZE, shuffle=False)
test_loader_shuffled = DataLoader(test_data_shuffled, batch_size=BATCH_SIZE, shuffle=False)
print(f"snapshot samples: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}, batch_size={BATCH_SIZE}")
print(f"one sample: nodes={train_data[0].num_nodes}, history_y_shape={tuple(train_data[0].history_y.shape)}, x_shape={tuple(train_data[0].x.shape)}")

## 6. Feature-Only Baselines With Lag Inputs

The tabular baselines receive the same current node features plus the five historical target values. They do not receive graph edges.

In [ ]:
def tabular_arrays(dataset: list[Data]) -> tuple[np.ndarray, np.ndarray]:
    X_parts, y_parts = [], []
    for data in dataset:
        X_parts.append(np.hstack([data.x.numpy(), data.history_y.numpy()]))
        y_parts.append(data.y_raw.numpy().ravel())
    return np.vstack(X_parts), np.concatenate(y_parts)


X_train_tab, y_train_tab = tabular_arrays(train_data)
X_val_tab, y_val_tab = tabular_arrays(val_data)
X_test_tab, y_test_tab = tabular_arrays(test_data)
results, predictions = [], {}

rf = RandomForestRegressor(n_estimators=RF_TREES, min_samples_leaf=5, max_samples=0.75, random_state=SEED, n_jobs=2)
rf.fit(X_train_tab, y_train_tab)
pred_rf = rf.predict(X_test_tab)
results.append({"model": "Random Forest + lags", **regression_metrics(y_test_tab, pred_rf)})
predictions["Random Forest + lags"] = pred_rf

try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(n_estimators=XGB_MAX_TREES, max_depth=4, learning_rate=0.04, subsample=0.9, colsample_bytree=0.9, objective="reg:squarederror", tree_method="hist", early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS, n_jobs=2, random_state=SEED)
    xgb.fit(X_train_tab, y_train_tab, eval_set=[(X_val_tab, y_val_tab)], verbose=False)
    best_iteration = getattr(xgb, "best_iteration", None)
    if best_iteration is not None:
        print(f"XGBoost best_iteration={best_iteration + 1} / {XGB_MAX_TREES}")
    pred_xgb = xgb.predict(X_test_tab)
    results.append({"model": "XGBoost + lags", **regression_metrics(y_test_tab, pred_xgb)})
    predictions["XGBoost + lags"] = pred_xgb
except Exception as exc:
    print(f"XGBoost unavailable ({exc}). Using sklearn HistGradientBoostingRegressor as a fallback.")
    hgb = HistGradientBoostingRegressor(max_iter=HGB_ITERATIONS, learning_rate=0.05, random_state=SEED)
    hgb.fit(X_train_tab, y_train_tab)
    pred_hgb = hgb.predict(X_test_tab)
    results.append({"model": "HistGradientBoosting + lags", **regression_metrics(y_test_tab, pred_hgb)})
    predictions["HistGradientBoosting + lags"] = pred_hgb

pd.DataFrame(results).sort_values("rmse")

## 7. GRU and STGNN Models

The non-spatial GRU learns from the previous five targets and current node features. The STGNN puts a GRU between two spatial message-passing stages:

`current features -> GraphSAGE -> GRU over lag targets + spatial context -> GraphSAGE -> prediction`

The shuffled-edge STGNN keeps the same lag values and features but breaks station relations.

In [ ]:
class NodeGRURegressor(torch.nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.history_gru = torch.nn.GRU(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.feature_encoder = torch.nn.Linear(in_dim, hidden_dim)
        self.out = torch.nn.Sequential(torch.nn.ReLU(), torch.nn.Dropout(0.15), torch.nn.Linear(hidden_dim * 2, hidden_dim), torch.nn.ReLU(), torch.nn.Linear(hidden_dim, 1))
    def forward(self, data: Data) -> torch.Tensor:
        _, h_last = self.history_gru(data.history_y.unsqueeze(-1))
        h_time = h_last[-1]
        h_feat = torch.relu(self.feature_encoder(data.x))
        return self.out(torch.cat([h_feat, h_time], dim=-1))


class STGraphSAGEGRURegressor(torch.nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.spatial_in = SAGEConv(in_dim, hidden_dim)
        self.history_gru = torch.nn.GRU(input_size=hidden_dim + 1, hidden_size=hidden_dim, batch_first=True)
        self.spatial_out = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = torch.nn.Dropout(0.15)
        self.out = torch.nn.Linear(hidden_dim, 1)
    def forward(self, data: Data) -> torch.Tensor:
        h_spatial = torch.relu(self.spatial_in(data.x, data.edge_index))
        h_repeated = h_spatial.unsqueeze(1).repeat(1, data.history_y.shape[1], 1)
        gru_input = torch.cat([data.history_y.unsqueeze(-1), h_repeated], dim=-1)
        _, h_last = self.history_gru(gru_input)
        h = self.dropout(h_last[-1])
        h = torch.relu(self.spatial_out(h, data.edge_index))
        return self.out(h)


def loader_loss(model: torch.nn.Module, loader: DataLoader, loss_fn: torch.nn.Module) -> float:
    model.eval()
    total_loss, total_nodes = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            loss = loss_fn(model(batch), batch.y)
            total_loss += float(loss.detach().cpu()) * batch.num_nodes
            total_nodes += batch.num_nodes
    return total_loss / max(total_nodes, 1)


def predict_loader(model: torch.nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(DEVICE)
            pred_scaled = model(batch).detach().cpu().numpy().ravel()
            preds.append(pred_scaled * y_std + y_mean)
    return np.concatenate(preds)


def train_neural_model(model: torch.nn.Module, train_loader: DataLoader, val_loader: DataLoader, test_loader: DataLoader, model_name: str, epochs: int = GNN_EPOCHS, lr: float = 0.002, patience: int = GNN_PATIENCE, log_every: int = 20) -> tuple[np.ndarray, pd.DataFrame]:
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = torch.nn.SmoothL1Loss()
    history, best_state, best_val_loss, stale_epochs = [], None, float("inf"), 0
    for epoch in range(epochs):
        model.train()
        running_loss, running_nodes = 0.0, 0
        for batch in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(batch), batch.y)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.detach().cpu()) * batch.num_nodes
            running_nodes += batch.num_nodes
        train_loss = running_loss / max(running_nodes, 1)
        val_loss = loader_loss(model, val_loader, loss_fn)
        history.append({"model": model_name, "epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})
        if epoch == 0 or (epoch + 1) % log_every == 0:
            print(f"{model_name:28s} epoch={epoch + 1:03d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
        if stale_epochs >= patience:
            break
    print(f"{model_name:28s} stopped_after={len(history)} best_val_loss={best_val_loss:.4f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    return predict_loader(model, test_loader), pd.DataFrame(history)


torch.manual_seed(SEED)
in_dim = train_data[0].x.shape[1]
pred_gru, hist_gru = train_neural_model(NodeGRURegressor(in_dim), train_loader, val_loader, test_loader, "GRU no graph")
results.append({"model": "GRU no graph", **regression_metrics(y_test_tab, pred_gru)})
predictions["GRU no graph"] = pred_gru

torch.manual_seed(SEED)
pred_stgnn, hist_stgnn = train_neural_model(STGraphSAGEGRURegressor(in_dim), train_loader, val_loader, test_loader, "STGNN GraphSAGE-GRU")
results.append({"model": "STGNN GraphSAGE-GRU", **regression_metrics(y_test_tab, pred_stgnn)})
predictions["STGNN GraphSAGE-GRU"] = pred_stgnn

torch.manual_seed(SEED)
pred_stgnn_shuffled, hist_stgnn_shuffled = train_neural_model(STGraphSAGEGRURegressor(in_dim), train_loader_shuffled, val_loader_shuffled, test_loader_shuffled, "STGNN shuffled edges")
results.append({"model": "STGNN shuffled edges", **regression_metrics(y_test_tab, pred_stgnn_shuffled)})
predictions["STGNN shuffled edges"] = pred_stgnn_shuffled

training_history = pd.concat([hist_gru, hist_stgnn, hist_stgnn_shuffled], ignore_index=True)
pd.DataFrame(results).sort_values("rmse")

## 8. Compare Results

Lower RMSE and MAE are better. Higher R2 is better. Compare `GRU no graph` with `STGNN GraphSAGE-GRU`, then compare the real STGNN with `STGNN shuffled edges`.

In [ ]:
metrics_table = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
display(metrics_table)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric, color in zip(axes, ["rmse", "mae", "r2"], ["#507dbc", "#3f7f93", "#8a6f3d"]):
    ordered = metrics_table.sort_values(metric, ascending=(metric != "r2"))
    ax.barh(ordered["model"], ordered[metric], color=color)
    ax.set_title(metric.upper())
    ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
for model_name, model_history in training_history.groupby("model"):
    ax.plot(model_history["epoch"], model_history["val_loss"], label=model_name)
ax.set_title("Neural model validation loss")
ax.set_xlabel("epoch")
ax.set_ylabel("SmoothL1 loss on standardized target")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

score_by_model = metrics_table.set_index("model")
diagnostics = []
if {"STGNN GraphSAGE-GRU", "STGNN shuffled edges"}.issubset(score_by_model.index):
    diagnostics.append({"question": "Do real edges matter after adding lag targets?", "comparison": "STGNN shuffled RMSE - STGNN real RMSE", "delta_rmse": score_by_model.loc["STGNN shuffled edges", "rmse"] - score_by_model.loc["STGNN GraphSAGE-GRU", "rmse"], "interpretation": "positive means real station edges help"})
if {"GRU no graph", "STGNN GraphSAGE-GRU"}.issubset(score_by_model.index):
    diagnostics.append({"question": "Does graph context help beyond temporal lags?", "comparison": "GRU RMSE - STGNN real RMSE", "delta_rmse": score_by_model.loc["GRU no graph", "rmse"] - score_by_model.loc["STGNN GraphSAGE-GRU", "rmse"], "interpretation": "positive means the STGNN improves on temporal lags alone"})
display(pd.DataFrame(diagnostics))

## 9. Interpretation Checklist

- If lagged tree models perform very well, the previous five target snapshots already carry strong temporal signal.
- If `STGNN GraphSAGE-GRU` beats `GRU no graph`, spatial message passing adds information beyond temporal autoregression.
- If `STGNN shuffled edges` is close to the real STGNN, the model may be relying mostly on lag targets and node features rather than meaningful station relations.
- The split uses embargo gaps. Do not randomly split node rows here, because the same graph snapshots would leak into train and test through lag windows.
- Increase `DATASET_FRACTION`, `RF_TREES`, `XGB_MAX_TREES`, or `GNN_EPOCHS` only after the default version runs end to end.